In [1]:
import sys
sys.path.append("/Users/shanesarosh/")
import os
os.environ["JAX_PLATFORMS"] = "cpu"  # <-- ADD THIS LINE
import chess
import chess.svg
from jax import random as jrandom
import numpy as np

In [2]:
from searchless_chess.src import tokenizer
from searchless_chess.src import training_utils
from searchless_chess.src import transformer
from searchless_chess.src import utils
from searchless_chess.src.engines import engine
from searchless_chess.src.engines import neural_engines

In [3]:
# @title Create the predictor.

policy = 'action_value'
num_return_buckets = 128

match policy:
  case 'action_value':
    output_size = num_return_buckets
  case 'behavioral_cloning':
    output_size = utils.NUM_ACTIONS
  case 'state_value':
    output_size = num_return_buckets
  case _:
    raise ValueError(f'Unknown policy: {policy}')

predictor_config = transformer.TransformerConfig(
    vocab_size=utils.NUM_ACTIONS,
    output_size=output_size,
    pos_encodings=transformer.PositionalEncodings.LEARNED,
    max_sequence_length=tokenizer.SEQUENCE_LENGTH + 2,
    num_heads=8,           # <-- CHANGE THIS TO 8
    num_layers=16,         # <-- CHANGED TO 16 FOR 270M
    embedding_dim=1024,    # <-- CHANGE THIS TO 1024
    apply_post_ln=True,
    apply_qk_layernorm=False,
    use_causal_mask=False,
)

predictor = transformer.build_transformer_predictor(config=predictor_config)

In [4]:
# @title Load the predictor parameters

checkpoint_dir = os.path.join(
    os.getcwd(),
    '../checkpoints/270M/',
)
dummy_params = predictor.initial_params(
    rng=jrandom.PRNGKey(0),
    targets=np.zeros((1, 1), dtype=np.uint32),
)
params = training_utils.load_parameters(
    checkpoint_dir=checkpoint_dir,
    params=dummy_params,
    use_ema_params=True,
    step=-1,
)

# ====== MEMORY BANDWIDTH HACK ======
import jax
# Cast all 32-bit weights down to 16-bit precision to double memory throughput
params = jax.tree_util.tree_map(lambda x: x.astype(jax.numpy.bfloat16), params)
# ===================================

In [5]:
# @title Create the engine

predict_fn = neural_engines.wrap_predict_fn(predictor, params, batch_size=1)
_, return_buckets_values = utils.get_uniform_buckets_edges_values(
    num_return_buckets
)

neural_engine = neural_engines.ENGINE_FROM_POLICY[policy](
    return_buckets_values=return_buckets_values,
    predict_fn=predict_fn,
    temperature=0.005,
)

In [ ]:
# @title Play a move with the agent

board = chess.Board()
best_move = neural_engine.play(board)
print(f'Best move: {best_move}')

In [ ]:
# @title Compute the win percentages for all legal moves

board = chess.Board()
results = neural_engine.analyse(board)
buckets_log_probs = results['log_probs']

# Compute the expected return.
win_probs = np.inner(np.exp(buckets_log_probs), return_buckets_values)
sorted_legal_moves = engine.get_ordered_legal_moves(board)

print(board.fen())
print(f'Win percentages:')
for i in np.argsort(win_probs)[::-1]:
  print(f'  {sorted_legal_moves[i].uci()} -> {100*win_probs[i]:.1f}%')

In [6]:
import asyncio
import websockets
import json
import chess
import chess.polyglot
import logging
import threading
import time

# Suppress connection noise in terminal
logging.getLogger('websockets').setLevel(logging.ERROR)

# === 1. LOAD THE OPENING BOOK ===
try:
    reader = chess.polyglot.open_reader("opening_book.bin")
    print("📚 gm2001 Opening Book loaded successfully!")
except FileNotFoundError:
    print("⚠️ No gm2001.bin found. Ensure the file is in the same directory.")
    reader = None

async def handle_bridge(websocket):
    print("🌐 Extension connected via WebSocket pipeline!")
    
    try:
        async for message in websocket:
            data = json.loads(message)
            req_type = data.get('type', 'play')
            
            try:
                # === 2. THE UI KILL SWITCH ===
                if req_type == 'reset':
                    print("♻️ Manual Reset Triggered! Ready for new game.")
                    continue

                incoming_fen = data.get('fen', '')
                board = chess.Board(incoming_fen)
                
                if board.is_game_over() or len(list(board.legal_moves)) == 0:
                    continue

                if req_type == 'play':
                    # Start the primary timer
                    t0 = time.perf_counter()
                    
                    # === 3. OPENING BOOK CHECK (Zero Compute) ===
                    if reader:
                        try:
                            # find() trips an IndexError if position is out of book
                            book_entry = reader.find(board) 
                            await websocket.send(json.dumps({"best_move": book_entry.move.uci()}))
                            t_book = time.perf_counter()
                            print(f"📚 BOOK HIT! Streaming GM theory instantly.")
                            print(f"⏱️ LATENCY: Book Hit in {(t_book - t0) * 1000:.2f} ms\n")
                            continue
                        except IndexError:
                            pass # Out of book, fall through to live calculation
                    
                    # Stop the book check timer
                    t1 = time.perf_counter()
                    
                    # === 4. LIVE TENSOR CALCULATION ===
                    best_move = neural_engine.play(board)
                    
                    # Stop the tensor math timer
                    t2 = time.perf_counter()
                    
                    # === 5. NETWORK DISPATCH ===
                    await websocket.send(json.dumps({"best_move": best_move.uci()}))
                    
                    # Stop the network timer
                    t3 = time.perf_counter()
                    
                    # === 6. PROFILING OUTPUT ===
                    book_check_time = (t1 - t0) * 1000
                    inference_time = (t2 - t1) * 1000
                    network_out_time = (t3 - t2) * 1000
                    total_time = (t3 - t0) * 1000
                    
                    print(f"🧠 Move Computed:")
                    print(f"   ↳ Book Check: {book_check_time:.2f} ms")
                    print(f"   ↳ NPU Inference: {inference_time:.2f} ms")
                    print(f"   ↳ Socket Send:  {network_out_time:.2f} ms")
                    print(f"   ===============================")
                    print(f"   ⚡ TOTAL LATENCY: {total_time:.2f} ms\n")
                    
            except ValueError:
                pass
                
    except websockets.exceptions.ConnectionClosed:
        print("🔌 Extension disconnected from pipeline.")

async def start_server():
    async with websockets.serve(handle_bridge, "127.0.0.1", 8000):
        print("🚀 Stream Engine running on ws://127.0.0.1:8000")
        await asyncio.Future()

def run_server_in_thread():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_server())

threading.Thread(target=run_server_in_thread, daemon=True).start()

📚 gm2001 Opening Book loaded successfully!
🚀 Stream Engine running on ws://127.0.0.1:8000
🌐 Extension connected via WebSocket pipeline!
🌐 Extension connected via WebSocket pipeline!
🧠 Move Computed:
   ↳ Book Check: 1.06 ms
   ↳ NPU Inference: 3350.99 ms
   ↳ Socket Send:  0.10 ms
   ⚡ TOTAL LATENCY: 3352.14 ms

🧠 Move Computed:
   ↳ Book Check: 2.01 ms
   ↳ NPU Inference: 3766.12 ms
   ↳ Socket Send:  0.08 ms
   ⚡ TOTAL LATENCY: 3768.21 ms

🧠 Move Computed:
   ↳ Book Check: 1.23 ms
   ↳ NPU Inference: 3633.85 ms
   ↳ Socket Send:  0.08 ms
   ⚡ TOTAL LATENCY: 3635.17 ms

🧠 Move Computed:
   ↳ Book Check: 0.41 ms
   ↳ NPU Inference: 4174.93 ms
   ↳ Socket Send:  0.08 ms
   ⚡ TOTAL LATENCY: 4175.41 ms

🧠 Move Computed:
   ↳ Book Check: 1.60 ms
   ↳ NPU Inference: 4473.18 ms
   ↳ Socket Send:  0.09 ms
   ⚡ TOTAL LATENCY: 4474.88 ms

🧠 Move Computed:
   ↳ Book Check: 0.83 ms
   ↳ NPU Inference: 5013.30 ms
   ↳ Socket Send:  0.22 ms
   ⚡ TOTAL LATENCY: 5014.35 ms

🧠 Move Computed:
   ↳ Book

In [ ]:
%pip install flask flask-cors

In [ ]:
!pip install websockets

In [ ]:
import os
print(os.getcwd())